In [1]:


!pip -q install sentence-transformers tiktoken

import hashlib, time

import numpy as np

import tiktoken

from sentence_transformers import SentenceTransformer

def header(title):

    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")

header("SETUP: load embedding models")

models = {

    "minilm": SentenceTransformer("all-MiniLM-L6-v2"),

    "bge":    SentenceTransformer("BAAI/bge-base-en-v1.5"),

}

for name, m in models.items():

    print(f"{name}: dims={m.get_sentence_embedding_dimension()}  max_tokens={m.max_seq_length}")

header("EXPERIMENT A: what similarity captures (look at 0-1, 2-3, 4-5)")

sents = [

    "What is the overdraft fee?",                            # 0

    "How much do I pay if my account goes negative?",       # 1  same meaning, no shared words

    "The overdraft fee is waived for premium accounts.",    # 2

    "The overdraft fee is charged for premium accounts.",   # 3  OPPOSITE of 2

    "We had a picnic on the river bank.",                   # 4

    "I opened a savings account at the bank.",              # 5

]

for name, m in models.items():

    e = m.encode(sents, normalize_embeddings=True)

    sim = e @ e.T

    print(f"\n{name}\n{np.round(sim, 2)}")

    print(f"  0 vs 1 (same meaning):     {sim[0, 1]:.2f}")

    print(f"  2 vs 3 (opposite meaning): {sim[2, 3]:.2f}")

    print(f"  4 vs 5 (two 'banks'):      {sim[4, 5]:.2f}")

header("EXPERIMENT B: silent truncation past max_tokens")

for name, m in models.items():

    filler = "Branches open at 9am and close at 5pm on weekdays. " * 50

    a = filler + "The overdraft fee is $35 per item."

    b = filler + "Our downtown branch has a coffee shop."

    n = len(m.tokenizer(a)["input_ids"])

    ea, eb = m.encode([a, b], normalize_embeddings=True)

    print(f"{name}: text={n} tokens, limit={m.max_seq_length}, cosine(a, b)={float(ea @ eb):.4f}")

header("EXPERIMENT C: bge query prefix")

bge = models["bge"]

docs = ["The overdraft fee is $35 per item.",

        "Overdraft protection links your savings account to checking.",

        "Branches open at 9am on weekdays."]

d = bge.encode(docs, normalize_embeddings=True)

q = "What is the overdraft fee?"

for label, query in [("no prefix", q),

                     ("prefix   ", "Represent this sentence for searching relevant passages: " + q)]:

    print(label, np.round(d @ bge.encode(query, normalize_embeddings=True), 3))

header("EXPERIMENT D: throughput + 10M-chunk estimate")

texts = [f"Txn {i}: debit card purchase at merchant {i % 50}, amount ${i % 500}.00" for i in range(2000)]

for name, m in models.items():

    t0 = time.perf_counter()

    m.encode(texts, batch_size=64, normalize_embeddings=True)

    rate = len(texts) / (time.perf_counter() - t0)

    dims = m.get_sentence_embedding_dimension()

    print(f"{name}: {rate:,.0f} texts/s | 10M chunks ≈ {10_000_000 / rate / 3600:,.1f} h "

          f"| storage ≈ {10_000_000 * dims * 4 / 1e9:.1f} GB (float32)")

header("EXPERIMENT E: same text, different tokenizers (warm-up Q2)")

sample = "The overdraft fee is $35 per item under Reg E 12 CFR 1005.17. ACCT_ID=0012345"

counts = {

    "tiktoken cl100k_base (GPT-4)":  len(tiktoken.get_encoding("cl100k_base").encode(sample)),

    "tiktoken o200k_base (GPT-4o)":  len(tiktoken.get_encoding("o200k_base").encode(sample)),

}

for name, m in models.items():

    counts[f"{name} tokenizer"] = len(m.tokenizer(sample, add_special_tokens=False)["input_ids"])

for k, v in counts.items():

    print(f"{k:32} {v} tokens")

header("EXPERIMENT F: a complete retriever in ~10 lines")

retr = models["minilm"]

corpus = ["Overdraft fee: $35 per item.",

          "The overdraft fee is waived for premium accounts.",

          "Wire transfers cost $25 each.",

          "Branches open at 9am on weekdays.",

          "Overdraft protection links your savings account to checking."]

corpus_vecs = retr.encode(corpus, normalize_embeddings=True)        # INDEXING: once

def retrieve(question, k=2):

    qv = retr.encode(question, normalize_embeddings=True)            # QUERY: embed question

    scores = corpus_vecs @ qv                                        # cosine vs every chunk

    top = np.argsort(scores)[::-1][:k]

    return [(corpus[i], round(float(scores[i]), 3)) for i in top]

for question in ["How much do I pay if my account goes negative?",

                 "Do premium customers pay overdraft fees?",

                 "When does the branch open?"]:

    print(f"\nQ: {question}")

    for text, score in retrieve(question):

        print(f"   {score}  {text}")

header("EXERCISE (W1D2): write chunk_by_tokens yourself")

enc = tiktoken.get_encoding("cl100k_base")

def chunk_by_tokens(text, max_tokens=256, overlap=32):

    # TODO: your code here. Delete the raise when done.

    raise NotImplementedError

def run_chunker_tests():

    text = "The overdraft fee is $35 per item. " * 200

    chunks = chunk_by_tokens(text, max_tokens=100, overlap=20)

    n = len(enc.encode(text))

    expected = -(-(n - 20) // (100 - 20))   # ceil((N - O) / (S - O))

    print(f"chunks: {len(chunks)} | formula predicts: {expected}")

    print("all chunks <= 100 tokens:", all(len(enc.encode(c)) <= 100 for c in chunks))

    print("empty text -> []:        ", chunk_by_tokens("", 100, 20) == [])

    print("short text -> 1 chunk:   ", len(chunk_by_tokens("short text", 100, 20)) == 1)

    print("overlap=0 rebuilds text: ", "".join(chunk_by_tokens(text, 100, 0)) == text)

    try:

        chunk_by_tokens(text, 100, 100)

        print("overlap>=size raises:     False  <- should raise ValueError")

    except ValueError:

        print("overlap>=size raises:     True")

try:

    run_chunker_tests()

except NotImplementedError:

    print("Not written yet: implement chunk_by_tokens above, then re-run the cell.")



SETUP: load embedding models


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


bge
[[1.   0.65 0.76 0.81 0.38 0.57]
 [0.65 1.   0.58 0.62 0.32 0.57]
 [0.76 0.58 1.   0.91 0.4  0.6 ]
 [0.81 0.62 0.91 1.   0.37 0.59]
 [0.38 0.32 0.4  0.37 1.   0.49]
 [0.57 0.57 0.6  0.59 0.49 1.  ]]
  0 vs 1 (same meaning):     0.65
  2 vs 3 (opposite meaning): 0.91
  4 vs 5 (two 'banks'):      0.49

EXPERIMENT B: silent truncation past max_tokens


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (663 > 512). Running this sequence through the model will result in indexing errors


minilm: text=663 tokens, limit=256, cosine(a, b)=1.0000
bge: text=663 tokens, limit=512, cosine(a, b)=1.0000

EXPERIMENT C: bge query prefix
no prefix [0.845 0.734 0.455]
prefix    [0.772 0.655 0.378]

EXPERIMENT D: throughput + 10M-chunk estimate


/tmp/ipykernel_1214/4130267667.py:111: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dims = m.get_sentence_embedding_dimension()


minilm: 114 texts/s | 10M chunks ≈ 24.4 h | storage ≈ 15.4 GB (float32)
bge: 15 texts/s | 10M chunks ≈ 189.7 h | storage ≈ 30.7 GB (float32)

EXPERIMENT E: same text, different tokenizers (warm-up Q2)
tiktoken cl100k_base (GPT-4)     28 tokens
tiktoken o200k_base (GPT-4o)     28 tokens
minilm tokenizer                 29 tokens
bge tokenizer                    29 tokens

EXPERIMENT F: a complete retriever in ~10 lines

Q: How much do I pay if my account goes negative?
   0.471  Overdraft fee: $35 per item.
   0.417  The overdraft fee is waived for premium accounts.

Q: Do premium customers pay overdraft fees?
   0.822  The overdraft fee is waived for premium accounts.
   0.698  Overdraft fee: $35 per item.

Q: When does the branch open?
   0.74  Branches open at 9am on weekdays.
   0.106  Overdraft protection links your savings account to checking.

EXERCISE (W1D2): write chunk_by_tokens yourself
Not written yet: implement chunk_by_tokens above, then re-run the cell.


In [17]:
header("EXERCISE (W1D2): write chunk_by_tokens yourself")

enc = tiktoken.get_encoding("cl100k_base")

def chunk_by_tokens(text, max_tokens=256, overlap=32):

    if overlap >= max_tokens:
      raise ValueError("overlap must be smaller than max_tokens")
    ids = enc.encode(text)
    chunks = []
    step = max_tokens - overlap
    for start in range(0,len(ids),step):
      window = ids[start:start+max_tokens]
      chunks.append(enc.decode(window))
      if start + max_tokens >= len(ids):
        break

    return chunks

def run_chunker_tests():

    text = "The overdraft fee is $35 per item. " * 200

    chunks = chunk_by_tokens(text, max_tokens=100, overlap=20)

    n = len(enc.encode(text))

    expected = -(-(n - 20) // (100 - 20))   # ceil((N - O) / (S - O))

    print(f"chunks: {len(chunks)} | formula predicts: {expected}")

    print("all chunks <= 100 tokens:", all(len(enc.encode(c)) <= 100 for c in chunks))

    print("empty text -> []:        ", chunk_by_tokens("", 100, 20) == [])

    print("short text -> 1 chunk:   ", len(chunk_by_tokens("short text", 100, 20)) == 1)

    print("overlap=0 rebuilds text: ", "".join(chunk_by_tokens(text, 100, 0)) == text)

    try:

        chunk_by_tokens(text, 100, 100)

        print("overlap>=size raises:     False  <- should raise ValueError")

    except ValueError:

        print("overlap>=size raises:     True")

try:

    run_chunker_tests()

except NotImplementedError:

    print("Not written yet: implement chunk_by_tokens above, then re-run the cell.")



EXERCISE (W1D2): write chunk_by_tokens yourself
chunks: 25 | formula predicts: 25
all chunks <= 100 tokens: True
empty text -> []:         True
short text -> 1 chunk:    True
overlap=0 rebuilds text:  True
overlap>=size raises:     True


In [1]:

!pip -q install sentence-transformers tiktoken pyarrow

import hashlib, os, time
from datetime import datetime, timezone
import numpy as np, pandas as pd, tiktoken
from sentence_transformers import SentenceTransformer

def header(t):
    print(f"\n{'=' * 72}\n{t}\n{'=' * 72}")

enc = tiktoken.get_encoding("cl100k_base")
minilm = SentenceTransformer("all-MiniLM-L6-v2")
bge = SentenceTransformer("BAAI/bge-base-en-v1.5")

# =====================================================================
# PART 1 (Day 3): incremental embedding pipeline. Write the 2 TODOs.
# =====================================================================
MODEL, MODEL_NAME, MODEL_VERSION = minilm, "all-MiniLM-L6-v2", "v1"
MAX_TOKENS, OVERLAP, BATCH_SIZE = 120, 20, 32
DIM = MODEL.get_sentence_embedding_dimension()

def make_doc(title, facts, sections=8):
    return " ".join(f"{title} section {i}: {f}" for i in range(1, sections + 1) for f in facts)

docs_v1 = {
    "fee_schedule": make_doc("Fee schedule", ["The overdraft fee is $35 per item.",
                                              "Wire transfers cost $25 each.",
                                              "Premium accounts have no monthly fee."]),
    "overdraft_policy": make_doc("Overdraft policy", ["Customers must opt in to overdraft coverage for debit card transactions.",
                                                      "At most three overdraft fees are charged per day."]),
    "branch_hours": make_doc("Branch hours", ["Branches open at 9am and close at 5pm on weekdays.",
                                              "Saturday hours are 9am to noon."]),
}

def chunk_by_tokens(text, max_tokens=256, overlap=32):
    if overlap >= max_tokens:
        raise ValueError("overlap must be smaller than max_tokens")
    ids = enc.encode(text)
    chunks = []
    step = max_tokens - overlap
    for start in range(0, len(ids), step):
        chunks.append(enc.decode(ids[start:start + max_tokens]))
        if start + max_tokens >= len(ids):
            break
    return chunks

# ---------- TODO 1 ----------
def make_chunk_records(doc_id, text):
    """One dict per chunk: chunk_id, doc_id, chunk_index, text, token_count, content_hash."""
    records = []
    for i,chunk in enumerate(chunk_by_tokens(text,MAX_TOKENS,OVERLAP)):
        records.append({
            "chunk_id": f"{doc_id}:{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "text": chunk,
            "token_count": len(enc.encode(chunk)),
            "content_hash": hashlib.sha256(chunk.encode()).hexdigest(),
        })
    return records

# ---------- TODO 2 ----------
def plan_changes(records, manifest):
    """records: current chunk dicts. manifest: {chunk_id: content_hash} from previous runs.
    Return (to_embed, unchanged, deleted_ids)."""
    to_embed = []
    unchanged = []
    current_ids = set()
    for r in records:
        current_ids.add(r.get('chunk_id'))
        if (r.get('content_hash') != manifest.get(r['chunk_id'])):
            to_embed.append(r)
        else:
            unchanged.append(r)
    deleted_ids = set(manifest) - current_ids
    return to_embed,unchanged,deleted_ids

def embed_batches(texts):
    if not texts:
        return np.zeros((0, DIM), dtype=np.float32)
    return MODEL.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True).astype(np.float32)

def validate(records, vectors):
    errors = []
    if vectors.shape != (len(records), DIM):
        errors.append(f"shape {vectors.shape} != ({len(records)}, {DIM})")
    if np.isnan(vectors).any():
        errors.append("NaN in embeddings")
    if len(vectors) and not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3):
        errors.append("vectors not normalised")
    ids = [r["chunk_id"] for r in records]
    if len(ids) != len(set(ids)):
        errors.append("duplicate chunk_ids")
    for r in records:
        n = len(MODEL.tokenizer(r["text"])["input_ids"])
        if n > MODEL.max_seq_length:
            errors.append(f"{r['chunk_id']}: {n} model tokens > limit {MODEL.max_seq_length}")
        if not r["text"].strip():
            errors.append(f"{r['chunk_id']}: empty text")
    if errors:
        raise ValueError("Validation failed:\n  " + "\n  ".join(errors))

def run_pipeline(docs, store, manifest, label):
    header(f"RUN: {label}")
    t0 = time.perf_counter()
    records = [r for doc_id, text in docs.items() for r in make_chunk_records(doc_id, text)]
    to_embed, unchanged, deleted_ids = plan_changes(records, manifest)
    t1 = time.perf_counter()
    vectors = embed_batches([r["text"] for r in to_embed])
    embed_s = time.perf_counter() - t1
    validate(to_embed, vectors)
    now = datetime.now(timezone.utc).isoformat()
    for r, v in zip(to_embed, vectors):
        store[r["chunk_id"]] = {**r, "embedding": v, "model": MODEL_NAME,
                                "model_version": MODEL_VERSION, "embedded_at": now}
        manifest[r["chunk_id"]] = r["content_hash"]
    for cid in deleted_ids:
        store.pop(cid, None)
        manifest.pop(cid, None)
    tokens = sum(r["token_count"] for r in to_embed)
    rate = f"{len(to_embed) / embed_s:,.0f} chunks/s" if to_embed else "nothing to embed"
    print(f"docs={len(docs)} chunks={len(records)} embedded={len(to_embed)} "
          f"unchanged={len(unchanged)} deleted={len(deleted_ids)}")
    print(f"tokens embedded={tokens} | embed time={embed_s:.2f}s | {rate}")
    print(f"store size={len(store)} | total time={time.perf_counter() - t0:.2f}s")

def test_todos():
    header("PART 1 - TEST TODO 1: make_chunk_records")
    recs = make_chunk_records("demo", docs_v1["fee_schedule"])
    print("chunks:", len(recs))
    print("keys ok:", set(recs[0]) == {"chunk_id", "doc_id", "chunk_index", "text", "token_count", "content_hash"})
    print("first ids:", [r["chunk_id"] for r in recs[:3]], "(expect demo:0, demo:1, demo:2)")
    again = make_chunk_records("demo", docs_v1["fee_schedule"])
    print("deterministic hashes:", [r["content_hash"] for r in recs] == [r["content_hash"] for r in again])
    print("all <= MAX_TOKENS:", all(r["token_count"] <= MAX_TOKENS for r in recs))

    header("PART 1 - TEST TODO 2: plan_changes")
    current = [{"chunk_id": "d:0", "content_hash": "h0"},
               {"chunk_id": "d:1", "content_hash": "h1-NEW"},
               {"chunk_id": "d:2", "content_hash": "h2"}]
    old = {"d:0": "h0", "d:1": "h1", "d:9": "h9"}
    emb, same, gone = plan_changes(current, old)
    print("to_embed: ", [r["chunk_id"] for r in emb], "(expect ['d:1', 'd:2'])")
    print("unchanged:", [r["chunk_id"] for r in same], "(expect ['d:0'])")
    print("deleted:  ", sorted(gone), "(expect ['d:9'])")

def part1():
    test_todos()
    store, manifest = {}, {}
    run_pipeline(docs_v1, store, manifest, "1. first load (everything is new)")
    run_pipeline(docs_v1, store, manifest, "2. re-run, nothing changed (should embed 0)")
    docs_v2 = dict(docs_v1)
    docs_v2["fee_schedule"] = docs_v1["fee_schedule"].replace("$35", "$30")
    del docs_v2["branch_hours"]
    run_pipeline(docs_v2, store, manifest, "3. fee changed to $30 + branch_hours removed")

    header("PART 1 - SAVE: Parquet")
    df = pd.DataFrame(store.values())
    df.to_parquet("chunks.parquet", index=False)
    print(df[["chunk_id", "token_count", "content_hash", "model_version"]].to_string(max_colwidth=20))
    print(f"rows={len(df)} | file={os.path.getsize('chunks.parquet') / 1024:.0f} KB | "
          f"embedding dtype={df['embedding'].iloc[0].dtype}")

    header("PART 1 - USE: query the saved file")
    loaded = pd.read_parquet("chunks.parquet")
    mat = np.stack(loaded["embedding"].to_numpy())
    qv = MODEL.encode("How much is the overdraft fee?", normalize_embeddings=True)
    for i in np.argsort(mat @ qv)[::-1][:3]:
        print(f"{float(mat[i] @ qv):.3f}  {loaded.chunk_id[i]:<20} {loaded.text[i][:70]}...")

# =====================================================================
# PART 2 (Day 4): model comparison harness
# =====================================================================
def recall_at_k(ranked_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    return len(set(ranked_ids[:k]) & relevant) / len(relevant)

def reciprocal_rank(ranked_ids, relevant_ids):
    relevant = set(relevant_ids)
    for position, ranked_id in enumerate(ranked_ids, start=1):
        if ranked_id in relevant:
            return 1 / position
    return 0.0

corpus = {
    "c01": "Overdraft fee: $35 per item, charged when a transaction exceeds the available balance.",
    "c02": "Premium checking accounts are exempt from overdraft fees.",
    "c03": "A maximum of three overdraft fees can be charged per business day.",
    "c04": "Customers must opt in before overdraft coverage applies to ATM and one-time debit card transactions (Reg E, 12 CFR 1005.17).",
    "c05": "Overdraft protection transfers funds from a linked savings account; a $10 transfer fee applies.",
    "c06": "Domestic wire transfers cost $25; international wires cost $45.",
    "c07": "Incoming wire transfers are free for all account types.",
    "c08": "An NSF fee of $35 applies when an item is returned unpaid due to insufficient funds.",
    "c09": "Branches are open 9am to 5pm Monday through Friday and 9am to noon on Saturday.",
    "c10": "The monthly maintenance fee is $12, waived with a $1,500 minimum daily balance.",
    "c11": "ACH return code R01 means insufficient funds; R02 means the account is closed.",
    "c12": "Stop payment requests cost $30 per item and remain in effect for six months.",
}
corpus_ids, corpus_texts = list(corpus), list(corpus.values())

golden = [
    ("What happens if I spend more than I have in my account?",          {"c01"}, "paraphrase"),
    ("Can money move automatically from my savings to cover a shortfall?", {"c05"}, "paraphrase"),
    ("How much does it cost to send money abroad by wire?",              {"c06"}, "paraphrase"),
    ("Are you open on weekends?",                                        {"c09"}, "paraphrase"),
    ("What is the fee for a bounced check?",                             {"c08"}, "paraphrase"),
    ("What does 12 CFR 1005.17 require?",                                {"c04"}, "exact_code"),
    ("What is the NSF fee?",                                             {"c08"}, "exact_code"),
    ("What does ACH return code R02 mean?",                              {"c11"}, "exact_code"),
    ("What does R01 stand for?",                                         {"c11"}, "exact_code"),
    ("Do premium customers pay overdraft charges?",                      {"c02"}, "condition"),
    ("Is there a charge for receiving a wire?",                          {"c07"}, "condition"),
    ("How do I avoid the monthly account fee?",                          {"c10"}, "condition"),
    ("Which accounts don't pay for overdrafts?",                         {"c02"}, "condition"),
    ("How many overdraft fees can I get in one day?",                    {"c03"}, "number"),
    ("How long does a stop payment last?",                               {"c12"}, "number"),
    ("How much is the overdraft fee?",                                   {"c01"}, "number"),
]

BGE_PREFIX = "Represent this sentence for searching relevant passages: "
configs = [("minilm", minilm, ""), ("bge", bge, ""), ("bge+prefix", bge, BGE_PREFIX)]

def evaluate(name, model, prefix):
    doc_vecs = model.encode(corpus_texts, normalize_embeddings=True)
    t0 = time.perf_counter()
    q_vecs = [model.encode(prefix + q, normalize_embeddings=True) for q, _, _ in golden]
    latency_ms = (time.perf_counter() - t0) / len(golden) * 1000
    rows = []
    for (q, rel, qtype), qv in zip(golden, q_vecs):
        ranked = [corpus_ids[i] for i in np.argsort(doc_vecs @ qv)[::-1]]
        rows.append({"config": name, "type": qtype, "question": q,
                     "expected": ",".join(sorted(rel)), "got_top1": ranked[0], "top3": ranked[:3],
                     "r@1": recall_at_k(ranked, rel, 1), "r@3": recall_at_k(ranked, rel, 3),
                     "rr": reciprocal_rank(ranked, rel)})
    dims = model.get_sentence_embedding_dimension()
    return rows, {"config": name, "query_ms": round(latency_ms, 1), "dims": dims,
                  "GB_per_10M": round(10_000_000 * dims * 4 / 1e9, 1)}

def part2():
    all_rows, ops = [], []
    for name, model, prefix in configs:
        rows, op = evaluate(name, model, prefix)
        all_rows += rows
        ops.append(op)
    df = pd.DataFrame(all_rows)

    header("PART 2 - OVERALL (16 questions; 1 question = 6.25 points)")
    summary = df.groupby("config")[["r@1", "r@3", "rr"]].mean().rename(columns={"rr": "MRR"}).round(3)
    print(summary.join(pd.DataFrame(ops).set_index("config")).to_string())

    header("PART 2 - RECALL@1 BY QUESTION TYPE (averages hide failures)")
    print(df.pivot_table(index="type", columns="config", values="r@1", aggfunc="mean").round(2).to_string())

    header("PART 2 - MISSES AT RANK 1 (error analysis: read these!)")
    misses = df[df["r@1"] == 0][["config", "type", "question", "expected", "got_top1", "top3"]]
    print(misses.to_string(index=False) if len(misses) else "No misses.")

try:
    part1()
except NotImplementedError:
    print("\nPART 1 skipped: write make_chunk_records and plan_changes, then re-run.")
part2()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1667/1625808921.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = MODEL.get_sentence_embedding_dimension()



PART 1 - TEST TODO 1: make_chunk_records
chunks: 4
keys ok: True
first ids: ['demo:0', 'demo:1', 'demo:2'] (expect demo:0, demo:1, demo:2)
deterministic hashes: True
all <= MAX_TOKENS: True

PART 1 - TEST TODO 2: plan_changes
to_embed:  ['d:1', 'd:2'] (expect ['d:1', 'd:2'])
unchanged: ['d:0'] (expect ['d:0'])
deleted:   ['d:9'] (expect ['d:9'])

RUN: 1. first load (everything is new)
docs=3 chunks=10 embedded=10 unchanged=0 deleted=0
tokens embedded=1076 | embed time=0.89s | 11 chunks/s
store size=10 | total time=0.90s

RUN: 2. re-run, nothing changed (should embed 0)
docs=3 chunks=10 embedded=0 unchanged=10 deleted=0
tokens embedded=0 | embed time=0.00s | nothing to embed
store size=10 | total time=0.00s

RUN: 3. fee changed to $30 + branch_hours removed
docs=2 chunks=7 embedded=4 unchanged=3 deleted=3
tokens embedded=396 | embed time=0.24s | 16 chunks/s
store size=7 | total time=0.25s

PART 1 - SAVE: Parquet
             chunk_id  token_count         content_hash model_version
0   

/tmp/ipykernel_1667/1625808921.py:236: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dims = model.get_sentence_embedding_dimension()



PART 2 - OVERALL (16 questions; 1 question = 6.25 points)
              r@1    r@3    MRR  query_ms  dims  GB_per_10M
config                                                     
bge         0.812  0.938  0.880     103.0   768        30.7
bge+prefix  0.812  1.000  0.896     169.2   768        30.7
minilm      0.812  1.000  0.896      17.9   384        15.4

PART 2 - RECALL@1 BY QUESTION TYPE (averages hide failures)
config      bge  bge+prefix  minilm
type                               
condition   0.5         0.5     0.5
exact_code  1.0         1.0     1.0
number      1.0         1.0     1.0
paraphrase  0.8         0.8     0.8

PART 2 - MISSES AT RANK 1 (error analysis: read these!)
    config       type                                                question expected got_top1            top3
    minilm paraphrase What happens if I spend more than I have in my account?      c01      c02 [c02, c01, c05]
    minilm  condition                 Is there a charge for receiving a wire?      